# Section 3: Training the Model
*Owner: PPT lead*

**Needs from Section 2:** `X_train`, `X_test`, `y_train`, `y_test`
**Hands over:** `model`, plus the functions `train_model` and `predict`

The temporary cell below recreates the inputs so you can start before the Execution lead is done. **Delete it when merging.**

In [ ]:
# TEMPORARY STAND-IN: delete this whole cell after merging into the final notebook.
# It rebuilds what the Execution lead's section hands over, so you can work before it is ready.
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/Titanic-Dataset.csv")
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
df = pd.get_dummies(df, columns=["Embarked"], drop_first=True, dtype=int)

X = df.drop(columns="Survived")
y = df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score, log_loss

## 3.1 Function to train the model
`LogisticRegression()` creates an empty model and `fit()` learns from the training data. `max_iter=1000` allows enough learning steps to finish.

In [ ]:
def train_model(X_train, y_train):
    # Create a logistic regression model and train it on the training data
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    return model

model = train_model(X_train, y_train)
print("Model trained.")

## 3.2 Function to test (predict with) the model
`predict` gives the class (0 or 1). `predict_proba` gives the probability of each class.

In [ ]:
def predict(model, X):
    # Returns the predicted class and the probability of survival
    classes = model.predict(X)
    prob_survive = model.predict_proba(X)[:, 1]
    return classes, prob_survive

train_pred, _ = predict(model, X_train)
test_pred, test_prob = predict(model, X_test)

print("First 10 predicted classes:", test_pred[:10])
print("First 10 survival probabilities:", test_prob[:10].round(2))
print("Train accuracy:", round(accuracy_score(y_train, train_pred), 3))
print("Test accuracy: ", round(accuracy_score(y_test, test_pred), 3))

Train and test accuracy being close means the model is **not overfitting** (it is not just memorizing the training data). *(Comment on your own numbers here.)*

## 3.3 What did the model learn? (coefficients)
Each input column gets a **coefficient**. A positive coefficient pushes the prediction towards survival, a negative one pushes it away. Because the data is scaled, bigger absolute values mean a stronger effect.

In [ ]:
coef_table = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": model.coef_[0]
}).sort_values("coefficient")

print("Intercept:", model.intercept_.round(3))
display(coef_table)

plt.barh(coef_table["feature"], coef_table["coefficient"])
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Coefficient")
plt.title("Logistic regression coefficients")
plt.savefig("outputs/coefficients.png", bbox_inches="tight")
plt.show()

**Interpretation:** *(for example: Sex has a large positive coefficient, so being female increases the chance of survival; Pclass is negative, so a higher class number (3rd class) lowers it.)*

## 3.4 Training plot: loss going down
The sklearn `LogisticRegression` learns in one call, so it has no learning curve to draw. To **see** learning, we train the same kind of model (logistic loss) step by step with `SGDClassifier` and record the **log loss** after every epoch. Loss measures how wrong the predicted probabilities are, and lower is better. This model is only used for the plot; the main `model` above is the one we evaluate.

In [ ]:
sgd = SGDClassifier(loss="log_loss", learning_rate="constant", eta0=0.01, random_state=42)

train_losses, test_losses = [], []
for epoch in range(50):
    sgd.partial_fit(X_train, y_train, classes=np.unique(y_train))
    train_losses.append(log_loss(y_train, sgd.predict_proba(X_train)))
    test_losses.append(log_loss(y_test, sgd.predict_proba(X_test)))

plt.plot(range(1, 51), train_losses, label="Training loss")
plt.plot(range(1, 51), test_losses, label="Test loss")
plt.xlabel("Epoch")
plt.ylabel("Log loss")
plt.title("Loss decreases as the model learns")
plt.legend()
plt.savefig("outputs/training_loss.png", bbox_inches="tight")
plt.show()

print("Loss at epoch 1: ", round(train_losses[0], 3))
print("Loss at epoch 50:", round(train_losses[-1], 3))

**Interpretation:** *(the loss drops quickly at first and then flattens, meaning the model has learned what it can. Train and test curves staying close means it generalizes well.)*